# 13 — CIDER T-cell Score by Driver Gene Status: Systematic Analysis

Tests whether somatic driver mutations are associated with CIDER T-cell
infiltration, using linear regression adjusted for tumor type.

Two scopes:
1. **ICI-treated** — all ICI-treated patients (pan-cancer pool, `cpi_ready.rds`)
2. **Pan-cancer** — all T-cell group patients regardless of treatment (`ready.rds`)

For each driver feature the model is:
```
cider_log2 ~ driver_status [+ cohort_broad_go]
```
Tumor-type covariate (`cohort_broad_go`) is included when patients span
more than one tumor type. Drivers with fewer than `MIN_MUTATED` mutated
samples are excluded.

Regression results saved to `figs/driver_tcell_results.rds` / `.csv`.
The only figure this notebook produces is `figs/supplement_driver_tcell_fdr.png`
(FDR-adjusted Pan-cancer volcano + per-cohort forest plot, Pan-cancer and
ICI-treated side by side).

In [ ]:
library(dplyr)
library(tidyr)
library(purrr)
library(broom)
library(ggplot2)
library(ggrepel)
library(patchwork)


In [ ]:
args <- commandArgs(trailingOnly = FALSE)
script_path <- sub("--file=", "", args[grep("--file=", args)])
SCRIPT_DIR <- if (length(script_path) > 0) {
  normalizePath(dirname(script_path), mustWork = FALSE)
} else {
  getwd()
}

source(file.path(SCRIPT_DIR, "../helpers/map.r"))
source(file.path(SCRIPT_DIR, "../helpers/shortcuts.r"))
source(file.path(SCRIPT_DIR, "../helpers/config.r"))
source(file.path(SCRIPT_DIR, "../helpers/themes.r"))

## 1 — Data preparation

In [ ]:
extra_markers <- readRDS(paste0(CIDER_PROJ_DIR, "extra_markers.rds"))

# ICI-treated pool: pan-cancer pool, one row per patient
tcell_ici <- readRDS(paste0(CIDER_PROJ_DIR, "cpi_ready.rds")) |>
  dplyr::filter(gp == "T-cell", cohort_broad == "ICI-Treated") |>
  dplyr::select(sampleId, cohort_broad_go, cider_score_depth_adj) |>
  dplyr::inner_join(extra_markers, by = "sampleId") |>
  dplyr::mutate(cider_log2 = log2(cider_score_depth_adj + 1))

# Pan-cancer all patients: T-cell group, all treatments, deduplicated
tcell_pan <- readRDS(paste0(CIDER_PROJ_DIR, "ready.rds")) |>
  dplyr::filter(gp == "T-cell") |>
  dplyr::distinct(sampleId, .keep_all = TRUE) |>
  dplyr::select(sampleId, cohort_broad_go = cohort_pan_go, cider_score_depth_adj) |>
  dplyr::inner_join(extra_markers, by = "sampleId") |>
  dplyr::mutate(cider_log2 = log2(cider_score_depth_adj + 1))

# ICI per-cohort: tumor-type rows (cohort_broad != "ICI-Treated"),
# using cohort_broad as the stratification variable
tcell_ici_cohort <- readRDS(paste0(CIDER_PROJ_DIR, "cpi_ready.rds")) |>
  dplyr::filter(gp == "T-cell", cohort_broad != "ICI-Treated") |>
  dplyr::select(sampleId, cohort_broad_go = cohort_broad, cider_score_depth_adj) |>
  dplyr::inner_join(extra_markers, by = "sampleId") |>
  dplyr::mutate(cider_log2 = log2(cider_score_depth_adj + 1))

message("ICI-treated (pooled) N = ", nrow(tcell_ici))
message("ICI-treated (cohort) N = ", nrow(tcell_ici_cohort))
message("Pan-cancer N           = ", nrow(tcell_pan))

## 2 — Driver feature definitions

In [ ]:
# Discover ALL driver_* columns present in the data plus hla_loh.
# Low MIN_MUTATED threshold: only exclude drivers with near-zero prevalence.
MIN_MUTATED <- 3

driver_cols   <- sort(grep("^driver_", names(tcell_ici), value = TRUE))
driver_labels <- sub("^driver_", "", driver_cols)

all_features <- c(
  c("HLA LOH" = "hla_loh"),
  stats::setNames(driver_cols, driver_labels)
)

# Include a driver if it appears in >=MIN_MUTATED patients in either dataset
DRIVER_FEATURES <- all_features[vapply(all_features, function(col) {
  in_ici <- col %in% names(tcell_ici) && sum(tcell_ici[[col]] == 1L, na.rm = TRUE) >= MIN_MUTATED
  in_pan <- col %in% names(tcell_pan) && sum(tcell_pan[[col]] == 1L, na.rm = TRUE) >= MIN_MUTATED
  in_ici || in_pan
}, logical(1))]

message("Driver features included (n = ", length(DRIVER_FEATURES), "):")
print(DRIVER_FEATURES)

## 3 — Regression analysis

In [ ]:
# Linear regression: cider_log2 ~ driver_status [+ cohort_broad_go]
# cohort_broad_go covariate included for pooled analyses; omitted for per-cohort.
fit_driver_lm <- function(df, driver_col, driver_label, scope_label,
                          adjust_cohort = TRUE) {
  df_fit <- df |>
    dplyr::filter(!is.na(.data[[driver_col]]), !is.na(cider_log2)) |>
    dplyr::mutate(driver_status = as.integer(.data[[driver_col]]))

  n_mut <- sum(df_fit$driver_status == 1L, na.rm = TRUE)
  n_wt  <- sum(df_fit$driver_status == 0L, na.rm = TRUE)

  if (n_mut < MIN_MUTATED || n_wt < 5L) return(NULL)

  use_cohort <- adjust_cohort && dplyr::n_distinct(df_fit$cohort_broad_go) > 1L
  fmla <- if (use_cohort) {
    cider_log2 ~ driver_status + cohort_broad_go
  } else {
    cider_log2 ~ driver_status
  }

  fit <- tryCatch(lm(fmla, data = df_fit), error = function(e) NULL)
  if (is.null(fit)) return(NULL)

  broom::tidy(fit, conf.int = TRUE) |>
    dplyr::filter(term == "driver_status") |>
    dplyr::mutate(
      driver     = driver_label,
      driver_col = driver_col,
      scope      = scope_label,
      n_total    = nrow(df_fit),
      n_mutated  = n_mut,
      n_wt       = n_wt,
      cohort_adj = use_cohort
    ) |>
    dplyr::select(driver, driver_col, scope, n_total, n_mutated, n_wt,
                  estimate, std.error, conf.low, conf.high, statistic, p.value,
                  cohort_adj)
}

# Helper: run all drivers for a given dataframe + scope label
run_scope <- function(df, scope_label, adjust_cohort = TRUE) {
  purrr::imap(DRIVER_FEATURES,
    ~ fit_driver_lm(df, .x, .y, scope_label, adjust_cohort)) |>
    purrr::compact() |>
    dplyr::bind_rows()
}

# ── Pooled analyses ───────────────────────────────────────────────────────
res_pooled <- dplyr::bind_rows(
  run_scope(tcell_ici, "ICI-treated",  adjust_cohort = TRUE),
  run_scope(tcell_pan, "Pan-cancer",   adjust_cohort = TRUE)
)

# ── Per-cohort analyses ───────────────────────────────────────────────────
run_per_cohort <- function(df, prefix) {
  cohorts <- sort(unique(df$cohort_broad_go))
  purrr::map(cohorts, function(coh) {
    df_coh <- dplyr::filter(df, cohort_broad_go == coh)
    run_scope(df_coh, paste0(prefix, ": ", coh), adjust_cohort = FALSE)
  }) |> purrr::compact() |> dplyr::bind_rows()
}

res_cohort <- dplyr::bind_rows(
  run_per_cohort(tcell_ici_cohort, "ICI-treated"),
  run_per_cohort(tcell_pan, "Pan-cancer")
)

results <- dplyr::bind_rows(res_pooled, res_cohort) |>
  dplyr::mutate(neg_log10_p = -log10(p.value)) |>
  dplyr::arrange(scope, p.value)

message("Results (n = ", nrow(results), " rows, ",
        dplyr::n_distinct(results$scope), " scopes):")
print(results |> dplyr::count(scope))

In [ ]:
out_rds <- paste0(CIDER_PROJ_DIR, "figs/driver_tcell_results.rds")
out_csv <- paste0(CIDER_PROJ_DIR, "figs/driver_tcell_results.csv")

saveRDS(results, out_rds)
write.csv(results, out_csv, row.names = FALSE)

message("Saved: ", out_rds)
message("Saved: ", out_csv)

## 5 — Cohort ordering scaffolding (shared by the forest plot)

Builds `cohort_label_map` / `cohort_levels` / `n_cohorts`, reused by the
FDR-adjusted forest plot below. Derived directly from `tcell_pan` /
`tcell_ici_cohort` rather than from a per-gene-filtered results table --
the cohort list and per-cohort N's don't depend on which genes end up
significant, so there's nothing to filter by driver first.

In [ ]:
# ── Cohort ordering scaffolding (shared by the forest plot below) ────────────
# cohort_label_map / cohort_levels / n_cohorts describe every per-cohort
# stratum each driver was tested in, and the order they're displayed in --
# built directly from tcell_pan / tcell_ici_cohort.
pan_n_map <- tcell_pan |>
  dplyr::count(cohort_broad_go, name = "n_total") |>
  dplyr::rename(cohort = cohort_broad_go)
ici_n_map <- tcell_ici_cohort |>
  dplyr::count(cohort_broad_go, name = "n_total") |>
  dplyr::rename(cohort = cohort_broad_go)

# `name` is coerced to character immediately -- tcell_pan$cohort_broad_go
# (cohort_pan_go) is a FACTOR whose labels already have "(N = ...)" baked in
# (4_base.ipynb), so the early-return branch below fires for every
# pan-cancer cohort. Without this coercion, vapply()'s per-element call
# hands add_n() a length-1 FACTOR (iterating a factor with vapply/sapply
# does not unwrap it to character), and returning that factor straight back
# fails vapply(..., character(1))'s strict type check with "result is type
# 'integer'" (a factor's storage mode) -- same root cause as the double-log
# TMB bug flagged elsewhere in this codebase: a transform silently applied
# to the wrong representation of a value.
add_n <- function(name, n_map) {
  name <- as.character(name)
  if (grepl("\\(N", name)) return(name)  # already has N count
  n <- n_map$n_total[n_map$cohort == name]
  if (length(n) == 1L) paste0(name, " (N=", n, ")") else name
}

# as.character() applied AFTER sort() -- sorting the factor first preserves
# its intended level order (e.g. cohort_pan_go's size-based ordering from
# 4_base.ipynb); converting to character before vapply() just avoids the
# iteration gotcha above without changing which cohort comes first.
pan_cohorts_v <- as.character(sort(unique(tcell_pan$cohort_broad_go)))
ici_cohorts_v <- as.character(sort(unique(tcell_ici_cohort$cohort_broad_go)))

pan_cohorts_labelled <- vapply(pan_cohorts_v, add_n, character(1), n_map = pan_n_map)
ici_cohorts_labelled <- vapply(ici_cohorts_v, add_n, character(1), n_map = ici_n_map)

cohort_label_map <- c(
  setNames(pan_cohorts_labelled, pan_cohorts_v),
  setNames(ici_cohorts_labelled, ici_cohorts_v),
  "All cohorts (pooled)"     = paste0("All Pan-cancer (N=", nrow(tcell_pan), ")"),
  "All ICI-treated (pooled)" = paste0("All ICI-treated (N=", nrow(tcell_ici), ")")
)

cohort_levels <- c(
  paste0("All ICI-treated (N=", nrow(tcell_ici), ")"), ici_cohorts_labelled,
  " ",
  paste0("All Pan-cancer (N=", nrow(tcell_pan), ")"), pan_cohorts_labelled
)
n_cohorts <- length(cohort_levels)

In [ ]:
# Curated immune-related driver genes
IMMUNE_GENES <- c(
  # Antigen presentation / MHC
  "B2M", "HLA LOH", "HLA-A", "HLA-B", "HLA-C",
  # Immune checkpoint
  "CD274",
  # JAK-STAT / interferon signaling
  "JAK1", "JAK2", "JAK3",
  # DNA polymerase / hypermutation
  "POLE", "POLD1",
  # Immune exclusion drivers
  "STK11", "KEAP1", "EGFR", "PTEN", "IDH1",
  # MSI / chromatin
  "ARID1A", "RNF43",
  # Immune signaling / cytokine receptors
  "TNFRSF14", "CSF1R", "MYD88", "CRLF2", "STAT5B",
  # Transcription factors / other immune-related
  "GATA1", "CD44", "RAC1", "CDH1", "PTPN14", "ACVR1"
)

# ── FDR-adjusted version of supplement_driver_tcell ──────────────────────────

# Add FDR-adjusted p-values within each scope
results_fdr <- results |>
  dplyr::group_by(scope) |>
  dplyr::mutate(p.adj = p.adjust(p.value, method = "fdr")) |>
  dplyr::ungroup() |>
  dplyr::mutate(neg_log10_padj = -log10(p.adj))

# Genes for forest plot: Pan-cancer pooled, FDR p.adj < 0.05 (no minimum mutation-count filter)
sig_genes_fdr <- results_fdr |>
  dplyr::filter(scope == "Pan-cancer", p.adj < 0.05) |>
  dplyr::arrange(estimate) |>
  dplyr::pull(driver)

message("FDR-significant genes (FDR < 0.05): ",
        length(sig_genes_fdr), " - ", paste(sig_genes_fdr, collapse = ", "))

# ── Volcano: FDR-adjusted p on y-axis ────────────────────────────────────────
volcano_fdr_df <- results_fdr |>
  dplyr::filter(scope == "Pan-cancer", n_mutated >= 10) |>
  dplyr::mutate(
    immune_gene  = driver %in% IMMUNE_GENES,
    in_forest    = driver %in% sig_genes_fdr,
    # Non-significant points get the muted/reference grey; FDR-significant
    # points get the strong ink colour (literal grey30 -> COL_INK, grey65 -> COL_NEGATIVE).
    point_colour = dplyr::if_else(p.adj > 0.05, COL_NEGATIVE, COL_INK),
    point_size   = dplyr::if_else(in_forest, 3.0, 2.0),
    label        = dplyr::if_else(p.adj < 0.05 | immune_gene, driver, NA_character_),
    label_size   = dplyr::if_else(immune_gene | in_forest, 3.5, 2.8),
    label_alpha  = dplyr::if_else(p.adj < 0.05, 1.0, 0.45)
  )

fdr_thresh_y <- -log10(0.05)  # horizontal reference line at FDR = 0.05

p_volcano_fdr <- ggplot2::ggplot(
  volcano_fdr_df,
  ggplot2::aes(x = estimate, y = neg_log10_padj, colour = point_colour)
) +
  ggplot2::geom_vline(xintercept = 0, linetype = "solid",
                      colour = COL_NEGATIVE, linewidth = 0.3) +
  # FDR threshold is a dashed reference line -- COL_NEGATIVE per the project's
  # muted/reference-line convention (COL_OVERLAY's coral-red is reserved
  # exclusively for Figure 7's literature-ORR overlay).
  ggplot2::geom_hline(yintercept = fdr_thresh_y, linetype = "dashed",
                      colour = COL_NEGATIVE, linewidth = 0.45, alpha = 0.65) +
  ggplot2::annotate("text", x = Inf, y = fdr_thresh_y,
                    label = "FDR = 0.05", hjust = 1.1, vjust = -0.45,
                    size = 2.8, colour = COL_NEGATIVE, alpha = 0.85) +
  ggplot2::geom_point(ggplot2::aes(size = point_size), alpha = 0.7) +
  ggrepel::geom_text_repel(
    ggplot2::aes(label = label, size = label_size, alpha = label_alpha),
    max.overlaps = 25, box.padding = 0.3, show.legend = FALSE
  ) +
  ggplot2::scale_size_identity() +
  ggplot2::scale_alpha_identity() +
  ggplot2::scale_colour_identity() +
  go_theme +
  ggplot2::labs(
    title    = "Pan-Cancer Driver Gene Associations with CIDER T-cell Score",
    subtitle = paste0("N = ", nrow(volcano_fdr_df),
                      " drivers (≥10 events); BH FDR-adjusted p-value"),
    x        = "Δ CIDER T-cell Score (log2)",
    y        = "-log₁₀(FDR-adjusted p-value)"
  ) +
  ggplot2::theme(
    plot.title    = ggplot2::element_text(size = 17),
    plot.subtitle = ggplot2::element_text(hjust = 0.5, size = 10),
    # Explicit small right-margin keeps the y-axis title tight against the
    # panel — ggplot2's default title margin scales up with font size and
    # otherwise leaves a large gap for a 14pt title.
    axis.title.y  = ggplot2::element_text(size = 14, margin = ggplot2::margin(r = 6)),
    axis.title.x  = ggplot2::element_text(size = 11),
    axis.text.x   = ggplot2::element_text(size = 12),
    axis.text.y   = ggplot2::element_text(size = 10)
  )

set_plot_dims(7, 5); p_volcano_fdr


In [ ]:
# ── Forest plot (FDR version) + save combined ─────────────────────────────────
if (length(sig_genes_fdr) == 0) {
  message("No genes meet FDR < 0.05; skipping FDR forest plot.")
} else {
  forest_fdr_df <- results_fdr |>
    dplyr::filter(
      driver %in% sig_genes_fdr,
      grepl("^Pan-cancer|^ICI-treated", scope)
    ) |>
    dplyr::mutate(
      scope_group  = dplyr::if_else(grepl("^Pan-cancer", scope),
                                    "Pan-cancer", "ICI-treated"),
      cohort       = dplyr::case_when(
        scope == "Pan-cancer"  ~ "All cohorts (pooled)",
        scope == "ICI-treated" ~ "All ICI-treated (pooled)",
        TRUE ~ sub("^(Pan-cancer|ICI-treated): ", "", scope)
      ),
      is_pooled    = scope %in% c("Pan-cancer", "ICI-treated"),
      driver       = factor(driver, levels = sig_genes_fdr),  # ascending estimate, left-to-right — matches Panel A's x-axis direction
      point_colour = COL_NEGATIVE,
      # Full opacity only where FDR p.adj < 0.05 (pooled) or raw p < 0.1 (per-cohort)
      point_alpha  = dplyr::case_when(
        is.na(p.adj)       ~ 0.0,
        p.adj  < 0.05      ~ 1.0,
        p.value < 0.1      ~ 0.5,
        TRUE               ~ 0.2
      )
    ) |>
    dplyr::mutate(
      cohort = dplyr::recode(as.character(cohort), !!!cohort_label_map),
      cohort = factor(cohort, levels = rev(cohort_levels))
    )

  # Spacer rows to preserve gap between Pan-cancer and ICI-treated sections
  spacer_fdr_df <- dplyr::tibble(
    driver       = factor(levels(forest_fdr_df$driver),
                          levels = levels(forest_fdr_df$driver)),
    cohort       = factor(" ", levels = levels(forest_fdr_df$cohort)),
    estimate     = NA_real_, conf.low = NA_real_, conf.high = NA_real_,
    p.value      = NA_real_, p.adj    = NA_real_, neg_log10_padj = NA_real_,
    neg_log10_p  = NA_real_,
    is_pooled    = FALSE, scope_group = "spacer",
    point_colour = NA_character_, point_alpha = NA_real_,
    n_mutated    = NA_integer_
  )
  forest_fdr_df <- dplyr::bind_rows(forest_fdr_df, spacer_fdr_df)

  n_genes_fdr <- length(sig_genes_fdr)
  n_cols_fdr  <- min(13L, n_genes_fdr)  # one more than the non-FDR forest's 12-column cap
  fig_w_fdr   <- min(2.5 + n_cols_fdr * 1.9, 26) * 0.6

  # Height scales with the number of facet_wrap rows actually produced —
  # so tightening the FDR threshold (fewer genes -> fewer facet columns
  # -> possibly one less wrapped row) shrinks the figure accordingly.
  n_facet_rows_fdr <- ceiling(n_genes_fdr / n_cols_fdr)
  fig_h_fdr   <- (min(1.5 + n_cohorts * 0.25, 11) * 1.1) * n_facet_rows_fdr

  p_forest_fdr <- ggplot2::ggplot(
    forest_fdr_df,
    ggplot2::aes(
      x      = estimate,
      y      = cohort,
      colour = point_colour,
      alpha  = point_alpha,
      size   = is_pooled
    )
  ) +
    ggplot2::geom_vline(xintercept = 0, linetype = "dashed",
                        colour = COL_NEGATIVE, linewidth = 0.4) +
    ggplot2::geom_errorbarh(
      ggplot2::aes(xmin = conf.low, xmax = conf.high),
      height = 0.0, linewidth = 0.45
    ) +
    ggplot2::geom_point() +
    ggplot2::scale_x_continuous(breaks = c(-1, 0, 1)) +
    ggplot2::scale_y_discrete(drop = FALSE) +
    ggplot2::scale_colour_identity() +
    ggplot2::scale_alpha_identity() +
    ggplot2::scale_size_manual(
      values = c("FALSE" = 1.5, "TRUE" = 3.0), guide = "none"
    ) +
    ggplot2::facet_wrap(~ driver, ncol = n_cols_fdr, scales = "free_x") +
    go_theme +
    ggplot2::theme(
      plot.title      = ggplot2::element_text(size = 16),
      strip.text      = ggplot2::element_text(face = "bold", size = 9),
      axis.text.y     = ggplot2::element_text(size = 7),
      axis.text.x     = ggplot2::element_text(size = 12),
      axis.title.x    = ggplot2::element_text(size = 12),
      panel.spacing.x = ggplot2::unit(0.9, "lines"),
      plot.subtitle   = ggplot2::element_text(hjust = 0.5)
    ) +
    ggplot2::labs(
      title    = "CIDER T-cell Score: Driver Gene Effects across Cohorts (FDR < 0.05)",
      x        = "Δ CIDER T-cell Score (log2), estimate ± 95% CI",
      y        = NULL
    )

  set_plot_dims(fig_w_fdr, fig_h_fdr); p_forest_fdr

  # ── Combine and save ─────────────────────────────────────────────────────
  # Shift 10% of Panel A's height over to Panel A from Panel B (total
  # height unchanged) so Panel A reads a bit larger relative to the forest.
  heights_fdr   <- c(5 * 1.15, fig_h_fdr)
  height_shift  <- 0.10 * heights_fdr[1]
  heights_fdr[1] <- heights_fdr[1] + height_shift
  heights_fdr[2] <- heights_fdr[2] - height_shift

  combined_w_fdr <- max(7, fig_w_fdr) * 0.9 * 0.9 * 0.9 * 0.9 * 1.1  # 10% thinner (again), then 10% wider
  base_h_fdr    <- (5 + fig_h_fdr) * 1.15 * 1.1 * 0.6                # 40% shorter (prior baseline height)

  # 20% taller overall, with 90% of the added height going to Panel A and
  # only 10% to Panel B (most of the extra room goes to the volcano panel).
  added_h_fdr   <- 0.20 * base_h_fdr
  a_frac_fdr    <- heights_fdr[1] / sum(heights_fdr)
  b_frac_fdr    <- heights_fdr[2] / sum(heights_fdr)
  heights_fdr   <- c(
    base_h_fdr * a_frac_fdr + 0.9 * added_h_fdr,
    base_h_fdr * b_frac_fdr + 0.1 * added_h_fdr
  )
  combined_h_fdr <- sum(heights_fdr)                                 # = base_h_fdr * 1.20

  p_combined_fdr <- (
    (p_volcano_fdr + ggplot2::labs(tag = "A")) /
    (p_forest_fdr  + ggplot2::labs(tag = "B"))
  ) +
    patchwork::plot_layout(heights = heights_fdr)

  out_fdr <- paste0(CIDER_PROJ_DIR, "figs/supplement_driver_tcell_fdr.png")
  ggplot2::ggsave(out_fdr, p_combined_fdr,
                  width = combined_w_fdr, height = combined_h_fdr, dpi = 300)
  message("Saved: ", out_fdr)
}
